# Parallax — Quickstart Notebook

This notebook walks through a full analysis: load a chat export, run the pipeline, inspect stats, generate a dashboard, and compare runs.

Install parallax first:
```bash
pip install -e ".[dev]"
```

In [ ]:
import json
import random
import tempfile
from pathlib import Path
from datetime import datetime, timezone
from parallax.core import analyze
from parallax.core import languages as lang
from parallax.core import crosstabs as ct
from parallax.core import diff

## 1. Create synthetic chat data

Generate a small Discord-style export with mixed English and target-language messages.

In [ ]:
random.seed(42)
templates_zh = [
    "how to install this deepseek error",
    "openai api key how to get",
    "skills memory cron how to use",
    "this product is too expensive",
    "wants to connect feishu bot",
    "better than claude code",
    "docker install fails wsl",
    "is this the official site",
]
templates_en = [
    "how do I install this on macOS",
    "getting an error with the openai api key",
    "how to use skills memory and cron",
    "pricing is too expensive",
    "wants to connect this to Slack",
    "better than Claude Code honestly",
    "docker install fails on wsl",
    "is this the official site?",
]

messages = []
for i in range(100):
    is_zh = random.random() < 0.4
    tpl = random.choice(templates_zh if is_zh else templates_en)
    messages.append(
        {
            "id": str(i + 1),
            "author": {"id": f"u{i % 20}", "name": f"user_{i % 20}"},
            "timestamp": f"2026-01-{(i % 28) + 1:02d}T12:00:00+00:00",
            "content": tpl,
            "reactions": [],
            "attachments": [],
        }
    )

export = {
    "guild": {"name": "test"},
    "channel": {"id": "1", "name": "general"},
    "messages": messages,
}
print(
    f"Generated {len(messages)} messages from {len(set(m['author']['id'] for m in messages))} users"
)

## 2. Run the analysis pipeline

Load the export through the Discord adapter and run the full pipeline.

In [ ]:
tmpdir = Path(tempfile.mkdtemp())
export_path = tmpdir / "export.json"
export_path.write_text(json.dumps(export, ensure_ascii=False))

salt = analyze.ensure_salt(tmpdir / "salt.key")
msg_list = list(analyze.load_discord_export(export_path, salt))

lang_profile = lang.get_language_profile("zh")
region_profile = lang.get_region_profile("cn")

users, stats = analyze.run_pipeline(
    msg_list,
    language_profile=lang_profile,
    region_profile=region_profile,
    verbose=True,
)

print(f"Done! {stats['metadata']['total_messages']} messages analyzed.")
print(
    f"Users: {stats['users']['total']} ({stats['users']['target_primary']} target-primary)"
)

## 3. Inspect the stats

Look at what the pipeline detected: providers, friction signals, features, language distribution.

In [ ]:
print("=== Language Distribution ===")
for k, v in sorted(stats.get("language_distribution", {}).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

print("\n=== Provider Mentions ===")
for k, v in sorted(stats.get("providers", {}).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

print("\n=== Friction Signals ===")
for k, v in sorted(stats.get("friction_signals", {}).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

print("\n=== Feature Usage ===")
for k, v in sorted(stats.get("features", {}).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

## 4. Cross-tabulation

Run the cross-tabulation helper to see provider clusters x location proxy.

In [ ]:
from dataclasses import asdict

users_data = {}
for uid, u in users.items():
    d = asdict(u)
    d["channels"] = sorted(u.channels)
    d["hours_of_day"] = dict(u.hours_of_day)
    for key in (
        "providers",
        "competitors",
        "messaging",
        "features",
        "friction",
        "shadow_community",
    ):
        d[key] = dict(getattr(u, key))
    d["first_seen"] = u.first_seen.isoformat()
    d["last_seen"] = u.last_seen.isoformat()
    d["language_primary"] = u.language_primary()
    d["display_name"] = "<redacted>"
    users_data[uid] = d

users_list = list(users_data.values())
crosstabs, _ = ct.build_crosstabs(
    users_list,
    now=datetime(2026, 7, 1, tzinfo=timezone.utc),
    region_profile=region_profile,
)
print(f"Crosstabs generated with {len(crosstabs)} pivots")
print(f"Pivot keys: {list(crosstabs.keys())}")

## 5. Generate a dashboard

Create an interactive HTML dashboard from the stats.

In [ ]:
from parallax.streams.generate_dashboard import generate_dashboard

dashboard_path = tmpdir / "dashboard.html"
generate_dashboard([stats], dashboard_path, users=users_list, chat_jsonl=export_path)
print(f"Dashboard: {dashboard_path} ({dashboard_path.stat().st_size:,} bytes)")
print(f"Open: file://{dashboard_path}")

## 6. Compare two runs (diff)

Generate a second run with more messages and compare.

In [ ]:
# Add 50 more messages
for i in range(100, 150):
    is_zh = random.random() < 0.5
    tpl = random.choice(templates_zh if is_zh else templates_en)
    messages.append(
        {
            "id": str(i + 1),
            "author": {"id": f"u{i % 25}", "name": f"user_{i % 25}"},
            "timestamp": f"2026-02-{(i % 28) + 1:02d}T12:00:00+00:00",
            "content": tpl,
            "reactions": [],
            "attachments": [],
        }
    )

export2 = {
    "guild": {"name": "test"},
    "channel": {"id": "1", "name": "general"},
    "messages": messages,
}
export_path2 = tmpdir / "export2.json"
export_path2.write_text(json.dumps(export2, ensure_ascii=False))

msg_list2 = list(analyze.load_discord_export(export_path2, salt))
_, stats2 = analyze.run_pipeline(
    msg_list2, language_profile=lang_profile, region_profile=region_profile
)

print(diff.diff_stats(stats, stats2))

## 7. Incremental analysis

Use the state store to only process new messages.

In [ ]:
from parallax.core.state import StateStore

state_db = tmpdir / "state.db"
with StateStore(state_db) as store:
    store.mark_seen(msg_list[:100], str(export_path))
    new_msgs = store.filter_new(msg_list2, str(export_path))
    print(f"Total messages: {len(msg_list2)}")
    print(f"New messages: {len(new_msgs)}")

    _, new_stats = analyze.run_pipeline(
        new_msgs, language_profile=lang_profile, region_profile=region_profile
    )
    merged = analyze.merge_stats(stats, new_stats)
    print(f"Merged: {merged['metadata']['total_messages']} total messages")
    print(f"Incremental: {merged['metadata'].get('incremental_merge', False)}")

## Next steps

- Run `parallax init my-project` to scaffold a real project directory
- Edit `config/` YAML files to customize for your community
- Try different `--target-language` and `--region` settings
- Run the deep-analysis streams (B, C, D) for LLM-powered insights